### RAG Pipeline (Pdf document to Vector DB Pipeline)

In [26]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [27]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents=[]
    pdf_dir=Path(pdf_directory)
    
    #Find all pdf files recursively
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF Files to process")
    
    for pdf_file in pdf_files:
        print(f"\Processing: {pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()
            
            #Add source information to metadata
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'
                
                all_documents.extend(documents)
                print(f" Loaded {len(documents)} pages")
                
        except Exception as e:
            print(f"Error: {e}")
            
    print(f"\n Total documents loaded:{len(all_documents)}")
    return all_documents     

#Process all PDF's in the data directory
all_pdf_documents=process_all_pdfs("../data")           
        

<>:13: SyntaxWarning: invalid escape sequence '\P'
<>:13: SyntaxWarning: invalid escape sequence '\P'
C:\Users\OYV2\AppData\Local\Temp\ipykernel_34420\1775826993.py:13: SyntaxWarning: invalid escape sequence '\P'
  print(f"\Processing: {pdf_file.name}")


Found 2 PDF Files to process
\Processing: Agentic_AI_Course_Notes_and_Interview_QA.pdf
 Loaded 13 pages
 Loaded 13 pages
 Loaded 13 pages
 Loaded 13 pages
 Loaded 13 pages
 Loaded 13 pages
 Loaded 13 pages
 Loaded 13 pages
 Loaded 13 pages
 Loaded 13 pages
 Loaded 13 pages
 Loaded 13 pages
 Loaded 13 pages
\Processing: LangChain_Updated_Study_Guide.pdf
 Loaded 6 pages
 Loaded 6 pages
 Loaded 6 pages
 Loaded 6 pages
 Loaded 6 pages
 Loaded 6 pages

 Total documents loaded:205


In [28]:
all_pdf_documents

[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-08-28T13:59:48+00:00', 'source': '..\\data\\pdf_files\\Agentic_AI_Course_Notes_and_Interview_QA.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'source_file': 'Agentic_AI_Course_Notes_and_Interview_QA.pdf', 'file_type': 'pdf'}, page_content='Agentic AI - Complete Course Notes\nVideo: Complete Agentic AI Course (10+ Hours)\nKrishna - YouTube | youtube.com/watch?v=rV3HJ4LEZ7k\nBeginner-friendly notes in simple language with examples. Topics: LangChain, LangGraph, RAG, Vectorless RAG,\nDeep Agents, Guardrails, LLM Evaluation, LLM Gateways + Top Interview Questions.\nVideo Timestamps (Course Map)\n- 00:00:00  Introduction - GenAI vs Agentic AI overview\n- 00:02:31  LangChain Course (~2.5 hrs) - models, tools, agents, middleware\n- 02:35:12  LangGraph Course (~2.5 hrs) - StateGraph, nodes, edges, agents\n- 05:02:29  RAG Course (~2 hrs) - traditional + agentic RAG\n- 07:10:43  Vectorless RAG (~50 min) -

### Chunking
### Text splitting get into chunks

In [29]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
        
    return split_docs    
    

In [30]:
chunks=split_documents(all_pdf_documents)
chunks

Split 205 documents into 479 chunks

Example chunk:
Content: Agentic AI - Complete Course Notes
Video: Complete Agentic AI Course (10+ Hours)
Krishna - YouTube | youtube.com/watch?v=rV3HJ4LEZ7k
Beginner-friendly notes in simple language with examples. Topics: L...
Metadata: {'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-08-28T13:59:48+00:00', 'source': '..\\data\\pdf_files\\Agentic_AI_Course_Notes_and_Interview_QA.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'source_file': 'Agentic_AI_Course_Notes_and_Interview_QA.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'PyPDF', 'creator': 'PyPDF', 'creationdate': '2026-08-28T13:59:48+00:00', 'source': '..\\data\\pdf_files\\Agentic_AI_Course_Notes_and_Interview_QA.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1', 'source_file': 'Agentic_AI_Course_Notes_and_Interview_QA.pdf', 'file_type': 'pdf'}, page_content='Agentic AI - Complete Course Notes\nVideo: Complete Agentic AI Course (10+ Hours)\nKrishna - YouTube | youtube.com/watch?v=rV3HJ4LEZ7k\nBeginner-friendly notes in simple language with examples. Topics: LangChain, LangGraph, RAG, Vectorless RAG,\nDeep Agents, Guardrails, LLM Evaluation, LLM Gateways + Top Interview Questions.\nVideo Timestamps (Course Map)\n- 00:00:00  Introduction - GenAI vs Agentic AI overview\n- 00:02:31  LangChain Course (~2.5 hrs) - models, tools, agents, middleware\n- 02:35:12  LangGraph Course (~2.5 hrs) - StateGraph, nodes, edges, agents\n- 05:02:29  RAG Course (~2 hrs) - traditional + agentic RAG\n- 07:10:43  Vectorless RAG (~50 min) -

### Embedding and VectorStoreDB

In [31]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [39]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self,model_name:str="all-MiniLM-L6-v2"):
            self.model_name=model_name
            self.model=None
            self._load_model()    
        
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model:{self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension:{self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise    
    
    def generate_embeddings(self,texts:List[str])->np.ndarray:
        """
        Generate embedding for list of texts
        
        Args:
        texts:List of text strings to embed
        
        Return:
        numpy array of embedding with shape (len(texts),embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embedding for {len(texts)} texts...")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape:{embeddings.shape}")
        return embeddings
    
# initialize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model:all-MiniLM-L6-v2


d:\work_dsi\LangChain\VTRAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\OYV2\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4477.70it/s]


Model loaded successfully. Embedding dimension:384


C:\Users\OYV2\AppData\Local\Temp\ipykernel_34420\825110275.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension:{self.model.get_sentence_embedding_dimension()}")


### Vector DB

In [43]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name:Name of the ChromaDB collection
            persist_directory:Directory to persist the vector store
        """
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialize_store()    
        
    def _initialize_store(self):
        """Initialize ChromaDB client"""
        try:
            # Create persistent Chroma Client who refer the ChromaDB
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)
            
            #Get or create collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection:{self.collection_name}")
            print(f"Existing document in collection:{self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise    
        
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        """
        Add documents and their embedding to the vector store
        
        Args:
            documents:List of LangChain documents
            embedding:Corresponding embedding for the documents
        """
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents must watch number of Embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        
        #Prepare data for ChromaDB
        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]
        
        
        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            #Generate unique ID
            doc_id=f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            #Prepare metadata
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)
            
            #Document content
            documents_text.append(doc.page_content)
            
            #Embedding
            embeddings_list.append(embedding.tolist())
            
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
vectorstore=VectorStore()
vectorstore           
                             
        
        
                

Vector store initialized. Collection:pdf_documents
Existing document in collection:0


In [44]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]


### Generate the Embeddings
embeddings=embedding_manager.generate_embeddings(texts)


### Store into VectorDB
vectorstore.add_documents(chunks,embeddings)

Generating embedding for 479 texts...


Batches: 100%|██████████| 15/15 [00:13<00:00,  1.09it/s]


Generated embeddings with shape:(479, 384)
Adding 479 documents to vector store...
Successfully added 479 documents to vector store
Total documents in collection: 479


### Retriever Pipeline From VectorStore

In [100]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self,vector_store:VectorStore,embedding_manager:EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager:Manger for generating query embeddings
        """
        
        self.vector_store=vector_store
        self.embedding_manager=embedding_manager
        
    
    def retriever(self,query:str,top_k:int=5,score_threshold:float=0.0)->List[Dict[str,Any]]:
        """
        Retrieve relevant documents from query
        
        Args:
            query:THe search query
            top_k:Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents ans metadata
        """
        print(f"Retrieving documents for query:'{query}'")
        print(f"Top K: {top_k},Score threshold:{score_threshold}")
        
        #Generate query embeddings
        query_embedding=self.embedding_manager.generate_embeddings([query])[0]
        
        #Search in vector store
        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            
            # Process results
            retrieved_docs=[]
            
            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]
                
                for i,(doc_id,document,metadata,distance) in enumerate(zip(ids,documents,metadatas,distances)):
                    # Convert distance to similarity score (ChromaDB used cosine distance)
                    similarity_score=1-distance
                    
                    if similarity_score>=score_threshold:
                        retrieved_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'similarity_score':similarity_score,
                            'distance':distance,
                            'rank':i+1
                        })
                        
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else        :
                print("No documents found")
                
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []    
        
rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever        
                         

### Ask some question

In [101]:
rag_retriever.retriever("Why LangGraph is?")


Retrieving documents for query:'Why LangGraph is?'
Top K: 5,Score threshold:0.0
Generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.54it/s]

Generated embeddings with shape:(1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_a7075099_21',
  'content': "Thought: need weather. Action: get_weather('Pune'). Observation: sunny.\nQ: Why LangGraph when LangChain exists?\nA: LangGraph gives full control: custom nodes, branching, loops, checkpointing, HITL.\nEscalate to human when confidence < 70%.\nQ: What are Nodes and Edges?\nA: Node = step/function. Edge = flow between steps. Conditional edge = if/else routing.\nagent -> tools_condition -> tools OR END.\nQ: What is State in LangGraph?\nA: Shared data passed between nodes, usually message history.\nMessagesState = {'messages': [...]}\nPage 10",
  'metadata': {'page': 9,
   'creationdate': '2026-08-28T13:59:48+00:00',
   'file_type': 'pdf',
   'total_pages': 13,
   'source': '..\\data\\pdf_files\\Agentic_AI_Course_Notes_and_Interview_QA.pdf',
   'creator': 'PyPDF',
   'page_label': '10',
   'doc_index': 21,
   'content_length': 534,
   'producer': 'PyPDF',
   'source_file': 'Agentic_AI_Course_Notes_and_Interview_QA.pdf'},
  'similarity_score': 0.2081

### Integration VectorDB Context pipeline with LLM output

In [126]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()


### 1. Initialize the Groq LLM (set your Groq api key)
groq_api_key=os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="qwen/qwen3.6-27b",temperature=0.1,max_tokens=1024)

### 2. Simple RAG function:retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retriever(query, top_k=top_k)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    prompt = f"""Use the following context to answer the question concisely.
    If the answer is not in the context, say you don't know.
    Context:
    {context}
    Question:
    {query}
    Answer:"""
    # ✅ No .format() — use f-string only
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content  

In [127]:
answer = rag_simple("Why LangGraph is?", rag_retriever, llm)
print(answer)

Retrieving documents for query:'Why LangGraph is?'
Top K: 3,Score threshold:0.0
Generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  7.87it/s]

Generated embeddings with shape:(1, 384)


Retrieved 3 documents (after filtering)

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Context:** Repeated blocks of text containing Q&A pairs about LangGraph vs LangChain, Nodes/Edges, and State. Also includes a thought/action/observation pattern about weather in Pune.
   - **Question:** "Why LangGraph is?" (Note: The question is grammatically incomplete, but clearly refers to "Why LangGraph when LangChain exists?" or "Why use LangGraph?")
   - **Constraint:** "Use the following context to answer the question concisely. If the answer is not in the context, say you don't know."

2.  **Scan Context for Keywords:**
   - Keywords: "Why LangGraph", "LangGraph", "LangChain"
   - Found in context: "Q: Why LangGraph when LangChain exists? A: LangGraph gives full control: custom nodes, branching, loops, checkpointing, HITL."

3.  **Formulate Answer:**
   - Extract the exact answer from the context: "LangGraph gives full control: custom nodes, branching, loops, checkpoi

### Enhanced RAG Pipeline Features

In [117]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    
    results = retriever.retriever(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output["context"] = context
    return output



In [118]:
# Example usage:
result = rag_advanced("Why LangGraph is?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query:'Why LangGraph is?'
Top K: 3,Score threshold:0.1
Generating embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.00it/s]

Generated embeddings with shape:(1, 384)
Retrieved 3 documents (after filtering)


KeyError: "'messages'"